# Step 5: Logistic Regression Baseline

## 1. Objective

Establish a simple, interpretable baseline -- NOT the best possible model. Two variants are trained: an unweighted Logistic Regression (Model A) and a class-weighted one (Model B, `class_weight="balanced"`). Later steps (Random Forest, XGBoost, imbalance strategies, threshold optimization) will be compared against this baseline. The test set is evaluated here for completeness but is **never** used to choose between Model A and Model B -- that would defeat its purpose as a final, untouched evaluation set.

## 2. Load chronological datasets

In [ ]:
import sys
sys.path.append("..")

import json
import joblib
import numpy as np
import pandas as pd

from src.model_utils import (
    load_split_data,
    validate_model_features,
    find_constant_features,
    build_logistic_pipeline,
    train_model,
    evaluate_classifier,
    evaluate_at_k,
    extract_logistic_coefficients,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

data = load_split_data()
X_train, y_train = data["X_train"], data["y_train"]
X_val, y_val = data["X_validation"], data["y_validation"]
X_test, y_test = data["X_test"], data["y_test"]
feature_columns = data["feature_columns"]

print("X_train:", X_train.shape)
print("X_validation:", X_val.shape)
print("X_test:", X_test.shape)
print()
for name, y in [("train", y_train), ("validation", y_val), ("test", y_test)]:
    print(f"{name}: n={len(y):,}, fraud={int(y.sum()):,}, rate={y.mean()*100:.4f}%")

## 3. Verify feature/target separation

In [ ]:
print("Feature count:", len(feature_columns), "(expected 42)")
print("Identical columns across all 3 splits:", list(X_train.columns) == list(X_val.columns) == list(X_test.columns))
print("nameOrig in X_train:", "nameOrig" in X_train.columns)
print("nameDest in X_train:", "nameDest" in X_train.columns)
print("isFraud in X_train:", "isFraud" in X_train.columns)

## 4. Check numeric data quality

All 42 features must be numeric with no missing/infinite values before scaling. A non-numeric feature here would be a bug to STOP and report, not silently encode.

In [ ]:
for split_name, X in [("train", X_train), ("validation", X_val), ("test", X_test)]:
    checks = validate_model_features(X, feature_columns)
    print(f"{split_name}:")
    for k, v in checks.items():
        print(f"    {k}: {v}")
    assert all(checks.values()), f"Data quality check failed for {split_name}: {checks}"

print("\ndtypes (train):")
print(X_train.dtypes.value_counts())

constant_feats = find_constant_features(X_train)
print(f"\nConstant/near-constant features in train: {constant_feats if constant_feats else 'none'}")

## 5. Preprocessing and scaling

Logistic Regression is scale-sensitive. Each model uses a `Pipeline(StandardScaler -> LogisticRegression)` so the scaler is always fit and applied together with the model that depends on it. **The scaler is fit on `X_train` only** -- validation and test are only ever `.transform()`-ed by the pipeline's `.predict_proba()`, never seen during `.fit()`. Scaling the one-hot type columns alongside continuous columns is harmless for Logistic Regression (they just become a small number of scaled binary values) and keeps the pipeline simple.

## 6 & 7. Model A (unweighted) and Model B (class-weighted)

Training fraud rate is ~0.0816% -- extremely imbalanced. Model A trains without any imbalance handling (establishes the naive baseline). Model B uses `class_weight="balanced"` (inversely proportional to class frequency), showing how much that alone changes minority-class detection. SMOTE is explicitly out of scope for this step.

In [ ]:
results = {}
fitted = {}

for name, class_weight in [("model_a_unweighted", None), ("model_b_balanced", "balanced")]:
    pipe = build_logistic_pipeline(class_weight=class_weight, max_iter=1000, solver="lbfgs")
    train_info = train_model(pipe, X_train, y_train)
    print(f"{name}: training_time_sec={train_info['training_time_sec']:.2f}, n_iter={train_info['n_iter']}, converged={train_info['converged']}")
    fitted[name] = train_info

print("\nFit ONLY on (X_train, y_train) -- validation and test were not passed to .fit() for either model.")

## 8. Validation evaluation

Classification metrics (precision/recall/F1/confusion matrix) use the default **threshold = 0.5** -- a baseline operating point only, not chosen or optimized. ROC-AUC and PR-AUC are threshold-independent. Accuracy is deliberately not reported as a headline metric (Step 2/1 already established why it's meaningless here).

In [ ]:
for name, info in fitted.items():
    proba_val = info["pipeline"].predict_proba(X_val)[:, 1]
    info["proba_val"] = proba_val
    val_metrics = evaluate_classifier(y_val, proba_val, threshold=0.5)
    results.setdefault(name, {})["validation_metrics"] = val_metrics
    print(f"{name} -- VALIDATION @ threshold=0.5:")
    for k, v in val_metrics.items():
        print(f"    {k}: {v}")
    print()

## 9. Test evaluation -- FINAL TEST RESULTS

These are computed for completeness and reporting only. **They are not used to pick between Model A and Model B** -- that decision (when made, in a later step) will be based on validation.

In [ ]:
for name, info in fitted.items():
    proba_test = info["pipeline"].predict_proba(X_test)[:, 1]
    info["proba_test"] = proba_test
    test_metrics = evaluate_classifier(y_test, proba_test, threshold=0.5)
    results[name]["test_metrics"] = test_metrics
    print(f"{name} -- FINAL TEST RESULTS @ threshold=0.5:")
    for k, v in test_metrics.items():
        print(f"    {k}: {v}")
    print()

## 10. Precision@K / Recall@K

K in {100, 500, 1000, 5000, 10000}, all well below both split sizes (~980K validation, ~919K test), so nothing is skipped here. This ranks transactions by predicted probability and asks: if an investigator could only review the top K, how many would be genuine fraud (precision) and what fraction of all fraud would that catch (recall)?

In [ ]:
k_values = [100, 500, 1000, 5000, 10000]

for name, info in fitted.items():
    at_k_val = evaluate_at_k(y_val, info["proba_val"], k_values)
    at_k_test = evaluate_at_k(y_test, info["proba_test"], k_values)
    results[name]["at_k_validation"] = at_k_val.to_dict(orient="records")
    results[name]["at_k_test"] = at_k_test.to_dict(orient="records")
    print(f"{name} -- Precision@K / Recall@K (VALIDATION):")
    print(at_k_val.to_string(index=False))
    print(f"\n{name} -- Precision@K / Recall@K (TEST):")
    print(at_k_test.to_string(index=False))
    print()

## 11. Confusion matrices (threshold = 0.5)

In [ ]:
for name in fitted:
    for split in ["validation", "test"]:
        cm = results[name][f"{split}_metrics"]["confusion_matrix"]
        print(f"{name} -- {split}: TN={cm['tn']:,} FP={cm['fp']:,} FN={cm['fn']:,} TP={cm['tp']:,}")

These are reported factually. Model B catches far more fraud (higher TP, lower FN) at the cost of far more false positives -- neither is labeled "better" here; that tradeoff is exactly what threshold optimization (a later, separate step) exists to navigate deliberately rather than by accident of `class_weight`.

## 12. Coefficient interpretation

Coefficients are on the STANDARDIZED feature scale. `odds_ratio = exp(coefficient)`: the multiplicative change in the odds of fraud for a one-standard-deviation increase in that feature, **within this fitted model** -- not a causal claim about the real world.

In [ ]:
coef_tables = {}
for name, info in fitted.items():
    coefs = extract_logistic_coefficients(info["pipeline"], feature_columns)
    coef_tables[name] = coefs
    print(f"=== {name}: TOP 15 POSITIVE coefficients (associated with higher predicted log-odds of fraud) ===")
    print(coefs.sort_values("coefficient", ascending=False).head(15).to_string(index=False))
    print(f"\n=== {name}: TOP 15 NEGATIVE coefficients (associated with lower predicted log-odds of fraud) ===")
    print(coefs.sort_values("coefficient", ascending=True).head(15).to_string(index=False))
    extreme = coefs[coefs["coefficient"].abs() > 5]
    print(f"\nFeatures with |coefficient| > 5 (checked for numerical instability): {len(extreme)}")
    print()

**Notable finding:** `amount_exceeds_sender_balance` has a strongly *negative* coefficient in both models (~-4.1 and ~-4.7) -- transactions where the requested amount exceeds the sender's balance are associated with LOWER predicted fraud odds, which looks backwards at first. It makes sense once you recall Step 2/3's "drained account" pattern: simulated fraud in this dataset almost always withdraws EXACTLY the available balance (`amount == oldbalanceOrg`), so `amount_exceeds_sender_balance` is essentially never true for fraud, while it fires more often for noisy/incomplete legitimate balance records. This is a model behaving correctly on a dataset-specific pattern, not a bug -- and exactly the kind of thing worth calling out rather than accepting at face value. No coefficient exceeded |5| in either model, so nothing here suggests numerical instability.

## 13. Model comparison

Factual side-by-side -- no ranking, no winner declared.

In [ ]:
rows = []
for name, info in fitted.items():
    vm, tm = results[name]["validation_metrics"], results[name]["test_metrics"]
    rows.append({
        "model": name,
        "class_weight": "balanced" if "balanced" in name else None,
        "validation_precision": vm["precision"], "validation_recall": vm["recall"], "validation_f1": vm["f1"],
        "validation_roc_auc": vm["roc_auc"], "validation_pr_auc": vm["pr_auc"],
        "test_precision": tm["precision"], "test_recall": tm["recall"], "test_f1": tm["f1"],
        "test_roc_auc": tm["roc_auc"], "test_pr_auc": tm["pr_auc"],
    })
comparison = pd.DataFrame(rows)
comparison

In [ ]:
at_k_rows = []
for name in fitted:
    val_recs = {r["k"]: r for r in results[name]["at_k_validation"]}
    test_recs = {r["k"]: r for r in results[name]["at_k_test"]}
    for k in k_values:
        at_k_rows.append({
            "model": name, "k": k,
            "validation_precision_at_k": val_recs[k]["precision_at_k"],
            "validation_recall_at_k": val_recs[k]["recall_at_k"],
            "test_precision_at_k": test_recs[k]["precision_at_k"],
            "test_recall_at_k": test_recs[k]["recall_at_k"],
        })
at_k_comparison = pd.DataFrame(at_k_rows)
at_k_comparison

## 14. Limitations

- **Threshold = 0.5 is arbitrary here**, not chosen for any business objective -- Model A's low recall / high precision and Model B's high recall / low precision at this single threshold mostly reflect where `class_weight="balanced"` happens to push the decision boundary, not a considered tradeoff. Threshold optimization is a dedicated future step.
- **Test/train distribution shift**: test fraud rate (0.4361%) is ~5x train's (0.0816%) and ~7.6x validation's (0.0575%), because PaySim's legitimate transaction volume collapses in later simulated days while fraud counts stay roughly constant (documented in Steps 2 and 4). Test metrics are **not** directly comparable to train/validation prevalence, and are **not** described as "real-world" performance -- they reflect this specific simulated late period.
- **No model selection has occurred.** Both models are baselines; choosing between them (or moving to Random Forest/XGBoost) happens in later steps using validation results and the full modeling workflow, never using these test numbers as the decider.
- Logistic Regression assumes a linear (in log-odds) relationship between scaled features and the outcome -- it cannot capture interaction effects the way tree-based models can, which is exactly why this is a baseline rather than the final model.

## 15. Final baseline summary

In [ ]:
import os

joblib.dump(fitted["model_a_unweighted"]["pipeline"], "../models/logistic_regression_baseline.joblib")
joblib.dump(fitted["model_b_balanced"]["pipeline"], "../models/logistic_regression_balanced.joblib")
print("Saved models/logistic_regression_baseline.joblib and models/logistic_regression_balanced.joblib")

# Reload-consistency check (required sanity check 14/15)
for name, path in [
    ("model_a_unweighted", "../models/logistic_regression_baseline.joblib"),
    ("model_b_balanced", "../models/logistic_regression_balanced.joblib"),
]:
    reloaded = joblib.load(path)
    reloaded_proba = reloaded.predict_proba(X_val)[:, 1]
    identical = np.array_equal(fitted[name]["proba_val"], reloaded_proba)
    print(f"{name}: reloaded model produces IDENTICAL predictions: {identical}")
    assert identical

In [ ]:
metrics_out = {
    name: {
        "class_weight": "balanced" if "balanced" in name else None,
        "training_time_sec": fitted[name]["training_time_sec"],
        "n_iter": fitted[name]["n_iter"],
        "converged": fitted[name]["converged"],
        "validation_metrics": results[name]["validation_metrics"],
        "test_metrics": {**results[name]["test_metrics"], "label": "FINAL TEST RESULTS - not used for model selection"},
        "at_k_validation": results[name]["at_k_validation"],
        "at_k_test": results[name]["at_k_test"],
    }
    for name in fitted
}
metrics_out["_meta"] = {
    "threshold_used": 0.5,
    "note": "Threshold 0.5 is a baseline operating point only; threshold optimization is a separate future step.",
    "test_set_warning": (
        "Test fraud rate (0.4361%) is far higher than train (0.0816%) because PaySim's legitimate transaction "
        "volume collapses in later simulated days while fraud counts stay roughly constant. Test metrics are "
        "not comparable to train prevalence and are not used to choose between Model A and Model B."
    ),
}
with open("../results/logistic_regression_metrics.json", "w") as f:
    json.dump(metrics_out, f, indent=2)

coef_frames = []
for name, coefs in coef_tables.items():
    c = coefs.copy()
    c.insert(0, "model", name)
    coef_frames.append(c)
pd.concat(coef_frames, ignore_index=True).to_csv("../results/logistic_regression_coefficients.csv", index=False)

at_k_frames = []
for name in fitted:
    for split_name, records in [("validation", results[name]["at_k_validation"]), ("test", results[name]["at_k_test"])]:
        for rec in records:
            at_k_frames.append({"model": name, "split": split_name, **rec})
pd.DataFrame(at_k_frames).to_csv("../results/logistic_regression_precision_recall_at_k.csv", index=False)

print("Saved results/logistic_regression_metrics.json")
print("Saved results/logistic_regression_coefficients.csv")
print("Saved results/logistic_regression_precision_recall_at_k.csv")

**Summary:** two Logistic Regression baselines were trained on 4,463,587 train-period transactions (steps 1-323) with 42 causal, leakage-checked features, fit ONLY on train data (scaler included), evaluated on validation and test at a fixed 0.5 threshold. Model A (unweighted) is high-precision/low-recall; Model B (`class_weight="balanced"`) is high-recall/low-precision. Neither is declared the winner -- both are baselines for later comparison against Random Forest, XGBoost, dedicated imbalance handling, and deliberate threshold optimization. No model selection was made from test results, and no threshold was optimized.